In [1]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project= "charrell-personal")

In [3]:
#Initialize Map
Map = geemap.Map()
Map = geemap.Map(height="800px")

In [ ]:
from tgbs_rs.config import S2_INDEX_BANDS, START_DATE, END_DATE, DRIVE_FOLDER
from tgbs_rs.config_vis import S2_VIS_PARAMS
from tgbs_rs.preprocessing import get_s2_sr_collection
from tgbs_rs.utils import (build_default_sites_featurecollection, get_sites_geometry)
from tgbs_rs.summaries import (
    collection_to_site_timeseries, 
    build_annual_multiband_composites,
)
from tgbs_rs.export_rasters import export_table_to_drive

#### Define User Inputs

In [6]:
# Build site feature collection
sites_fc = build_default_sites_featurecollection()

# Get merged geometry for filtering image collections
sites_geom = get_sites_geometry(sites_fc)


ks_rehab_geom = sites_fc.filter(ee.Filter.eq("site_id", "ks_rehab")).geometry()
buda_geom = sites_fc.filter(ee.Filter.eq("site_id", "buda")).geometry()
gogoni_geom = sites_fc.filter(ee.Filter.eq("site_id", "gogoni")).geometry()
shimba_hills_geom = sites_fc.filter(ee.Filter.eq("site_id", "shimba_hills")).geometry()

In [7]:
# Build processed Sentinel-2 collection
s2_col = get_s2_sr_collection(
    aoi=sites_fc.bounds(),
    start_date=ee.Date(START_DATE),
    end_date=ee.Date(END_DATE),
    apply_water_masking=True,
)

In [8]:
annual_indices = build_annual_multiband_composites(
    collection=s2_col,
    bands=S2_INDEX_BANDS,
    start_date=ee.Date(START_DATE),
    end_date=ee.Date(END_DATE),
    composite_stat="median",
)

In [9]:
annual_site_summaries_fc = collection_to_site_timeseries(
    collection=annual_indices,
    sites_fc=sites_fc,
    bands=S2_INDEX_BANDS,
    reducer=ee.Reducer.mean(),
    scale=10,
    tile_scale=8,
)

In [10]:
export_table_to_drive(
    collection=annual_site_summaries_fc,
    description="tgbs_annual_s2_site_indices_2018_2025",
    folder=DRIVE_FOLDER,
    fileNamePrefix="tgbs_annual_s2_site_indices_2018_2025",
    fileFormat="CSV",
)

Export started: {'state': 'READY', 'description': 'tgbs_annual_s2_site_indices_2018_2025', 'priority': 100, 'creation_timestamp_ms': 1773991586251, 'update_timestamp_ms': 1773991586251, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'LVOQJUGQR22JRVQLLVCNH3PD', 'name': 'projects/charrell-personal/operations/LVOQJUGQR22JRVQLLVCNH3PD'}


In [ ]:
s2_rehab_2018 = get_s2_sr_collection(
    aoi=ks_rehab_geom,
    start_date=ee.Date("2018-02-01"),
    end_date=ee.Date("2018-12-31"),
    apply_water_masking=True
)
